In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BankingAnalytics").getOrCreate()

In [2]:
customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""

accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""

transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
"""

with open("bank_customers.csv","w") as f:
    f.write(customers_csv)

with open("accounts.csv","w") as f:
    f.write(accounts_csv)

with open("transactions.csv","w") as f:
    f.write(transactions_csv)

print("Files created")

Files created


In [4]:
branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""

with open("branches.csv", "w") as f:
    f.write(branches_csv)

print("branches.csv created")

branches.csv created


In [5]:
!ls

accounts.csv  bank_customers.csv  branches.csv	sample_data  transactions.csv


In [7]:
customer_profiles_json = """[
  {
    "customer_id": 1,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000011111"},
    "services": ["UPI", "Credit Card", "Net Banking"]
  },
  {
    "customer_id": 2,
    "name": "Sneha",
    "contact": {"email": "sneha@mail.com", "phone": "9000022222"},
    "services": ["UPI", "Debit Card"]
  },
  {
    "customer_id": 3,
    "name": "Arjun",
    "contact": {"email": "arjun@mail.com", "phone": "9000033333"},
    "services": ["Net Banking", "Loan"]
  }
]"""

with open("customer_profiles.json", "w") as f:
    f.write(customer_profiles_json)

print("customer_profiles.json created")

customer_profiles.json created


In [8]:
customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

In [11]:
customers.show()
accounts.show()
transactions.show()
branches.show()
customers.printSchema()
transactions.printSchema()
customers.count()
accounts.count()
transactions.count()
transactions.show(5)

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+

+----------+-----------+---------------

In [12]:
customers.select("name","city","account_type").show()
accounts.select("account_id","balance").show()
accounts.withColumnRenamed("balance","current_balance").show()
transactions.withColumnRenamed("txn_type","transaction_type").show()
customers.filter(col("city")=="Hyderabad").show()
customers.filter(col("age")>30).show()
customers.filter(col("account_type")=="Savings").show()
accounts.filter(col("balance")>100000).show()
transactions.filter(col("txn_type")=="Credit").show()
transactions.filter(col("amount")>20000).show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+--------

In [13]:
customers.orderBy("age").show()
customers.orderBy(col("age").desc()).show()
accounts.orderBy(col("balance").desc()).show()
accounts.orderBy(col("balance").desc()).limit(5).show()
accounts.orderBy(col("balance")).limit(5).show()
transactions.orderBy(col("amount").desc()).show()
transactions.orderBy(col("amount").desc()).limit(3).show()
transactions.orderBy("txn_date").show()
customers.orderBy("city","age").show()
transactions.orderBy(col("txn_date").desc()).limit(5).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+

+-----------+------+---------+---+-----

In [14]:
accounts.select(sum("balance")).show()
accounts.select(avg("balance")).show()
accounts.select(max("balance")).show()
accounts.select(min("balance")).show()
customers.groupBy("city").count().show()
customers.groupBy("account_type").count().show()
transactions.groupBy("txn_type").count().show()
transactions.filter(col("txn_type")=="Credit").select(sum("amount")).show()
transactions.filter(col("txn_type")=="Debit").select(sum("amount")).show()
transactions.select(avg("amount")).show()

+------------+
|sum(balance)|
+------------+
|     1055000|
+------------+

+-----------------+
|     avg(balance)|
+-----------------+
|87916.66666666667|
+-----------------+

+------------+
|max(balance)|
+------------+
|      200000|
+------------+

+------------+
|min(balance)|
+------------+
|       30000|
+------------+

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|    5|
|   Debit|    5|
+--------+-----+

+-----------+
|sum(amount)|
+-----------+
|     113000|
+-----------+

+-----------+
|sum(amount)|
+-----------+
|      87000|
+-----------+

+-----------+
|avg(amount)|
+-----------+
|    20000.0|
+-----------+



In [15]:
accounts.groupBy("branch").sum("balance").show()
accounts.groupBy("branch").avg("balance").show()
accounts.groupBy("branch").count().show()
transactions.groupBy("account_id").sum("amount").show()
transactions.groupBy("account_id").count().show()
transactions.groupBy("txn_type").sum("amount").show()
transactions.groupBy("txn_date").count().show()
customers.groupBy("city").avg("age").show()
customers.join(accounts, "customer_id") \
         .groupBy("account_type") \
         .sum("balance") \
         .show()
customers.groupBy("city","account_type").count().show()

+-----------------+------------+
|           branch|sum(balance)|
+-----------------+------------+
|      Delhi North|      205000|
|   Hyderabad Main|      285000|
|        Pune East|       30000|
|      Mumbai West|      245000|
|    Chennai South|       60000|
|Bangalore Central|      230000|
+-----------------+------------+

+-----------------+-----------------+
|           branch|     avg(balance)|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|        Pune East|    1|
|      Mumbai West|    2|
|    Chennai South|    1|
|Bangalore Central|    2|
+-----------------+-----+

+----------+-----------+
|a

In [16]:
cust_acc = customers.join(accounts, "customer_id")
cust_acc.show()

customers.join(accounts, "customer_id") \
         .select("name","city","branch","balance") \
         .show()

acc_txn = accounts.join(transactions, "account_id")
acc_txn.show()

accounts.join(transactions, "account_id") \
        .select("account_id","txn_type","amount","balance") \
        .show()

acc_branch = accounts.join(branches, "branch")
acc_branch.show()

accounts.join(branches, "branch") \
        .select("branch","region","manager","balance") \
        .show()

customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .show()

customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .select("name","city","txn_type","amount","txn_date") \
         .show()

customers.join(accounts, "customer_id") \
         .join(branches, "branch") \
         .join(transactions, "account_id") \
         .show()

customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .groupBy("name") \
         .sum("amount") \
         .show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [17]:
accounts.withColumn("balance_in_lakhs", col("balance")/100000).show()
accounts.withColumn("bank_name", lit("BotCampus Bank")).show()
accounts.withColumn("annual_service_fee", col("balance")*0.01).show()
accounts.withColumn("net_balance", col("balance") - (col("balance")*0.01)).show()
accounts.withColumn("is_high_balance", col("balance") > 100000).show()
transactions.withColumn("txn_amount_in_k", col("amount")/1000).show()
customers.withColumn("country", lit("India")).show()
customers.withColumn("customer_label", concat(col("name"), lit(" - "), col("city"))).show()

accounts.join(branches, "branch") \
        .withColumn("branch_label", concat(col("branch"), lit(" - "), col("region"))) \
        .show()

transactions.withColumn("risk_flag", col("amount") > 40000).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_in_lakhs|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|            0.85|
|      1002|          2|Bangalore Central| 120000|             1.2|
|      1003|          3|      Mumbai West|  45000|            0.45|
|      1004|          4|      Delhi North|  95000|            0.95|
|      1005|          5|    Chennai South|  60000|             0.6|
|      1006|          6|   Hyderabad Main| 150000|             1.5|
|      1007|          7|        Pune East|  30000|             0.3|
|      1008|          8|      Delhi North|  70000|             0.7|
|      1009|          9|Bangalore Central| 110000|             1.1|
|      1010|         10|      Mumbai West| 200000|             2.0|
|      1011|         11|   Hyderabad Main|  50000|             0.5|
|      1012|         12|      Delhi North|  4000

In [18]:
accounts.withColumn(
    "balance_category",
    when(col("balance") >= 100000, "High")
    .when(col("balance") >= 50000, "Medium")
    .otherwise("Low")
).show()

customers.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 40, "Adult")
    .otherwise("Senior")
).show()

transactions.withColumn(
    "txn_category",
    when(col("amount") >= 30000, "Large")
    .when(col("amount") >= 10000, "Medium")
    .otherwise("Small")
).show()

branches.withColumn(
    "region_priority",
    when(col("region") == "South", "High Priority")
    .when(col("region") == "North", "Medium Priority")
    .otherwise("Watch")
).show()

customers.withColumn(
    "account_label",
    when(col("account_type") == "Savings", "Retail")
    .otherwise("Business")
).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_category|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|          Medium|
|      1002|          2|Bangalore Central| 120000|            High|
|      1003|          3|      Mumbai West|  45000|             Low|
|      1004|          4|      Delhi North|  95000|          Medium|
|      1005|          5|    Chennai South|  60000|          Medium|
|      1006|          6|   Hyderabad Main| 150000|            High|
|      1007|          7|        Pune East|  30000|             Low|
|      1008|          8|      Delhi North|  70000|          Medium|
|      1009|          9|Bangalore Central| 110000|            High|
|      1010|         10|      Mumbai West| 200000|            High|
|      1011|         11|   Hyderabad Main|  50000|          Medium|
|      1012|         12|      Delhi North|  4000

In [19]:
customers = customers.withColumn("signup_date", to_date("signup_date"))
customers.show()

customers.withColumn("signup_year", year("signup_date")).show()
customers.withColumn("signup_month", month("signup_date")).show()
transactions = transactions.withColumn("txn_date", to_date("txn_date"))
transactions.show()
transactions.withColumn("txn_month", month("txn_date")).show()
transactions.groupBy("txn_date").count().show()

transactions.withColumn("txn_month", month("txn_date")) \
            .groupBy("txn_month") \
            .count() \
            .show()

customers.filter(col("signup_date") > "2023-06-01").show()
customers.withColumn("days_since_signup", datediff(current_date(), col("signup_date"))).show()
transactions.withColumn("days_since_txn", datediff(current_date(), col("txn_date"))).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+

+-----------+------+---------+---+-----

In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

cust_acc = customers.join(accounts, "customer_id")
cust_acc_txn = cust_acc.join(transactions, "account_id")
full_df = cust_acc.join(branches, "branch")

window_city = Window.partitionBy("city").orderBy(col("balance").desc())
cust_acc.withColumn("rank_in_city", rank().over(window_city)).show()

cust_acc.withColumn("row_num", row_number().over(window_city)) \
        .filter(col("row_num") == 1) \
        .show()

window_txn = Window.partitionBy("txn_type").orderBy(col("amount").desc())
transactions.withColumn("dense_rank", dense_rank().over(window_txn)).show()

window_acc = Window.partitionBy("account_id").orderBy(col("amount").desc())
transactions.withColumn("rank", rank().over(window_acc)) \
            .filter(col("rank") <= 2) \
            .show()

window_running = Window.partitionBy("account_id").orderBy("txn_date") \
                       .rowsBetween(Window.unboundedPreceding, 0)

transactions.withColumn("running_total", sum("amount").over(window_running)).show()

branch_balance = accounts.groupBy("branch").sum("balance")
window_branch = Window.orderBy(col("sum(balance)").desc())

branch_balance.withColumn("branch_rank", rank().over(window_branch)).show()

city_balance = customers.join(accounts, "customer_id") \
                        .groupBy("city") \
                        .sum("balance")

window_city_rank = Window.orderBy(col("sum(balance)").desc())

city_balance.withColumn("city_rank", rank().over(window_city_rank)).show()

cust_txn_total = customers.join(accounts, "customer_id") \
                          .join(transactions, "account_id") \
                          .groupBy("name") \
                          .sum("amount")

window_cust = Window.orderBy(col("sum(amount)").desc())

cust_txn_total.withColumn("customer_rank", rank().over(window_cust)).show()

city_txn = customers.join(accounts, "customer_id") \
                    .join(transactions, "account_id")

window_city_txn = Window.partitionBy("city").orderBy(col("amount").desc())

city_txn.withColumn("rank", rank().over(window_city_txn)) \
        .filter(col("rank") == 1) \
        .select("name","city","amount") \
        .show()

region_balance = customers.join(accounts, "customer_id") \
                          .join(branches, "branch")

window_region = Window.partitionBy("region").orderBy(col("balance").desc())

region_balance.withColumn("rank", rank().over(window_region)) \
              .filter(col("rank") == 1) \
              .select("name","region","balance") \
              .show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+------------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|rank_in_city|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+------------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|           1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|      1009|Bangalore Central| 110000|           2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|           1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|           1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      1008|      Delhi North|  70000|           2|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|      1012|      Delhi North|  40000|           3|
|         

In [21]:
from pyspark.sql.functions import *

profiles = spark.read.json("customer_profiles.json", multiLine=True)
profiles.show(truncate=False)

profiles.select(
    "customer_id",
    "name",
    col("contact.email").alias("email"),
    col("contact.phone").alias("phone")
).show()

profiles.select(
    "customer_id",
    "name",
    explode("services").alias("service")
).show()

profiles.select(
    "customer_id",
    "name",
    explode("services").alias("service")
).filter(col("service") == "UPI").show()

profiles.select(explode("services").alias("service")) \
        .groupBy("service") \
        .count() \
        .show()

+----------------------------+-----------+-----+-------------------------------+
|contact                     |customer_id|name |services                       |
+----------------------------+-----------+-----+-------------------------------+
|{rahul@mail.com, 9000011111}|1          |Rahul|[UPI, Credit Card, Net Banking]|
|{sneha@mail.com, 9000022222}|2          |Sneha|[UPI, Debit Card]              |
|{arjun@mail.com, 9000033333}|3          |Arjun|[Net Banking, Loan]            |
+----------------------------+-----------+-----+-------------------------------+

+-----------+-----+--------------+----------+
|customer_id| name|         email|     phone|
+-----------+-----+--------------+----------+
|          1|Rahul|rahul@mail.com|9000011111|
|          2|Sneha|sneha@mail.com|9000022222|
|          3|Arjun|arjun@mail.com|9000033333|
+-----------+-----+--------------+----------+

+-----------+-----+-----------+
|customer_id| name|    service|
+-----------+-----+-----------+
|          1|